In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [107]:
import torch

d_model = 384
block = 65

# **ROTATORY POSITIONAL ENCODING**

1. maintain position information across the layers
2. Can go beyond pre determined seq length
3. Put around the clock


In [108]:
g = torch.Generator().manual_seed(54321)

In [132]:

x = torch.randint(0, 65, (32, block, ), generator=g)
W = torch.randn((block, d_model), generator=g)


In [133]:
embs = W[x]
embs.shape

torch.Size([32, 65, 384])

In [61]:
# Numerical Identity is based on ideal assertions not on the groud truth

pos = torch.arange(block).view(-1, 1) # there are 64 tokens, which means there are 64 positions
i = torch.arange(d_model//2) # i is the input of positions (2i, 2i+1) where 2i, 2i+1 are even and odd demensions of model dimension (d_model)
angular_freq = 1e4 ** (-2 * i/d_model)

# this is the angular frequency, the rate of change angle with respect to dimension 2i normalized to d_model in range of 0 to 1
# distributing angular frequency between 1 to 10000.or 1 to 0.0001 or 0.0001 to 1 for for position say 63 it's like 0.00063 to 63
# Now 63e-4 to 63 is a range of angles(position specific range of angles) for every position in form i[0, d_model]
# Since we have the angles now we can distribute y-axis and x-axis as per these angles.
# So when we find sin of given range we are finding y-ordinates on y-axis
# and when we find cos of genve range we are finding x-ordination on x-axis
# there for every position of the token sequence we gets (x, y) --> coordinates representing d_model/2 
# needles pointing to every posible direction for that particular position of a token sequence
# it's like a representation on a clock for every position, like in clock for a time we need hour, min, sec hands
# similarly to represent a single position we can use d_model/2 needles, every possible arrangement of these needles
# can define new position.

angle  = pos * angular_freq

sins = angle.sin() # all y-ordinates 
coss = angle.cos() # all x-ordinates

ods, evns = embs[..., 0::2], embs[..., 1::2] # Now for every token at arbitrary position we have (x,y) pair co-ordinates representation of every d_model/2 needles representing
#that token. 

# Now when we pass it through the model and model my find to distinguish "man kills dog" or "dog kills man"
# So now we rotate every token representation to something aligned in a position system.
# hence getting position for every token


new_ods  =  ods * coss + evns * sins
new_evns = -ods * sins + evns * coss
embs_new =  torch.empty_like(embs)
embs_new[..., 0::2] = new_ods
embs_new[..., 1::2] = new_evns

In [68]:
(embs_new == embs).float().mean()

tensor(0.0156)

In [238]:
import torch
import torch.nn as nn
import torch.nn.functional as F



class RotatoryPositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len = 50000):
        super().__init__()
        pos = torch.arange(0, max_len).unsqueeze(1)
        ix = torch.arange(0, d_model//2).unsqueeze(0)
        omega = 1e4 ** (-2 * ix/d_model)
        angles = pos * omega
        self.register_buffer("cos", angles.cos())
        self.register_buffer("sin", angles.sin())

    def forward(self, x):
        T= x.shape[-2]
        cos, sin = self.cos[:T], self.sin[:T]

        X, Y = x[..., 0::2], x[..., 1::2] # x-ords, y-ords for every token
     
        Xn =  X * cos[:T] + Y * sin[:T] # [ cosQ sinQ ]  [X]
        Yn = -X * sin[:T] + Y * cos[:T] # [-sinQ cosQ ] [Y]
        
        out = torch.empty_like(x)
        out[..., 0::2] = Xn
        out[..., 1::2] = Yn

        return out
        

        

# **Grouped Query Attention**

In [201]:
B, T, C = embs.shape
H = 6
G = 2
R = H//G
hs = C//H

In [202]:
Wq = torch.randn(H, 1, C, hs)
Wk = torch.randn(G, 1, C, hs)
Wv = torch.randn(G, 1, C, hs)
Wo = torch.randn(H * hs, C)

Wq.shape, Wk.shape, Wv.shape, Wo.shape

(torch.Size([6, 1, 384, 64]),
 torch.Size([2, 1, 384, 64]),
 torch.Size([2, 1, 384, 64]),
 torch.Size([384, 384]))

In [203]:
q = torch.tensor([1, 2, 3])
q.shape

torch.Size([3])

In [212]:
Q = embs @ Wq # (32, 65, 384) @ (6, 1, 384, 64) --> 6, 32, 65, 64 (H, B, C, hs)
K = embs @ Wk # (32, 65, 384) @ (3, 2, 384, 64) --> 
K = torch.repeat_interleave(K, R, dim=0)
V = embs @ Wv
V = torch.repeat_interleave(V, R, dim=0)
Q.shape, K.shape, V.shape


(torch.Size([6, 32, 65, 64]),
 torch.Size([6, 32, 65, 64]),
 torch.Size([6, 32, 65, 64]))

In [237]:
mask = torch.tril(torch.ones(T, T))
score = Q @ K.transpose(-2, -1) / d_model
score = score.masked_fill(mask==0, float('-inf'))
probs = F.softmax(score, dim=-1) 
out = probs @ V
out = out.view(B, T, -1)
(out @ Wo).shape

torch.Size([32, 65, 384])

In [ ]:
class GroupedQueryAttention(nn.Module):

    def __init__(self, block_size, d_model, heads, groups):
        super().__init__()
        
        self.hs = d_model // heads
        self.rps = heads // groups
        
        self.Wqs = nn.ModuleList([nn.Linear(d_model, self.hs) for _ in range(heads)])
        self.Wks = nn.ModuleList([nn.Linear(d_model, self.hs) for _ in range(groups)])
        self.Wvs = nn.ModuleList([nn.Linear(d_model, self.hs) for _ in range(groups)])
        self.rope = RotatoryPositionalEncoding(self.hs, block_size)
        self.Wo = nn.Linear(heads * self.hs, d_model)

        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape # Batch, block_size, d_models (B, T, C)

        Q = self.rope(torch.stack([Wq(x) for Wq in self.Wqs]))
        K = torch.stack([Wk(x) for Wk in self.Wks])
        V = torch.stack([Wv(x) for Wv in self.Wvs])

        K = self.rope(torch.repeat_interleave(K, self.rps, dim=0))
        V = torch.repeat_interleave(V, self.rps, dim=0)

        score = Q @ K.transpose(-2, -1) / self.hs ** 0.5
        score = score.masked_fill(self.mask[:T,  :T]==0, float("-inf"))
        probs = F.softmax(score, dim=-1)
        out = probs @ V
        out = out.permute(1, 2, 0, 3).reshape(B, T, -1)
        out  = self.Wo(out)

        return out
        

        

